# Ablations: CE-only Protocol + GELU Third Seed
**Google Colab A100** — Runtime → Change runtime type → A100

**Two quick ablations (~45 min total):**

**Ablation A — CE-only (25 min):** Trains MLP-5 with cross-entropy for 600 epochs without switching to MSE. Shows NC1 does not reach < 0.01 with CE alone, fully justifying the two-phase protocol.

**Ablation B — GELU seeds 3+4 (20 min):** The existing GELU result has 2/3 seeds (seed 0 DNF). Tries seeds 3 and 4; stops at the first collapse. Removes the dagger from Table 2 and gives 3/3 seeds.

**Outputs:** `ce_only_ablation.csv`, `gelu_summary.csv`, `gelu_s3/4.csv`

In [1]:
import torch, torchvision, time
import torchvision.transforms as T
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from google.colab import files
torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
DEVICE = 'cuda'
assert torch.cuda.is_available()
print(f'GPU: {torch.cuda.get_device_name(0)}')


GPU: NVIDIA A100-SXM4-40GB


In [2]:
transform = T.Compose([T.ToTensor(), T.Normalize((0.1307,),(0.3081,))])
trainset  = torchvision.datasets.MNIST('/tmp/data', train=True,
                                       download=True, transform=transform)
testset   = torchvision.datasets.MNIST('/tmp/data', train=False,
                                       download=True, transform=transform)
train_loader = DataLoader(trainset, batch_size=512, shuffle=True,
                          num_workers=4, persistent_workers=True,
                          prefetch_factor=2, pin_memory=True)
test_loader  = DataLoader(testset,  batch_size=1024, shuffle=False,
                          num_workers=4, persistent_workers=True,
                          prefetch_factor=2, pin_memory=True)
print('MNIST loaded.')


100%|██████████| 9.91M/9.91M [00:00<00:00, 40.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.06MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.1MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.6MB/s]

MNIST loaded.


In [3]:
class MLP(nn.Module):
    def __init__(self, depth=5, width=512, act_cls=nn.ReLU, num_classes=10):
        super().__init__()
        layers = [nn.Flatten(), nn.Linear(784, width), act_cls()]
        for _ in range(depth-1):
            layers += [nn.Linear(width, width), act_cls()]
        self.body   = nn.Sequential(*layers)
        self.head   = nn.Linear(width, num_classes)
        self._feats = None
        self.body.register_forward_hook(
            lambda m, i, o: setattr(self, '_feats', o.detach()))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
    def forward(self, x): return self.head(self.body(x))
    def get_features(self, x): self(x); return self._feats
    def get_classifier_weights(self): return self.head.weight.detach()
print('MLP defined.')


MLP defined.


In [4]:
@torch.no_grad()
def compute_nc(model, loader, K=10):
    model.eval()
    fl, ll = [], []
    for x, y in loader:
        fl.append(model.get_features(x.to(DEVICE, non_blocking=True)))
        ll.append(y.to(DEVICE, non_blocking=True))
    H = torch.cat(fl); Y = torch.cat(ll)
    mu_G = H.mean(0)
    mu_c = torch.stack([H[Y==c].mean(0) for c in range(K)])
    M    = mu_c - mu_G
    Sw   = sum((H[Y==c]-mu_c[c]).T@(H[Y==c]-mu_c[c]) for c in range(K))/len(H)
    Sb   = M.T @ M / K
    nc1  = (torch.trace(Sw)/torch.trace(Sb).clamp(1e-10)).item()
    Mn   = F.normalize(M, dim=1)
    cos  = Mn @ Mn.T
    mask = ~torch.eye(K, dtype=torch.bool, device=DEVICE)
    nc2  = (cos[mask]-(-1./(K-1))).abs().mean().item()
    Wn   = F.normalize(model.get_classifier_weights().to(DEVICE), dim=1)
    nc3  = (1-(Mn*Wn).sum(1).mean()).item()
    return {'nc1':nc1,'nc2':nc2,'nc3':nc3,'feat_norm':H.norm(dim=1).mean().item()}

def evaluate(model, loader):
    model.eval(); correct=total=0
    with torch.no_grad():
        for x, y in loader:
            x,y = x.to(DEVICE,non_blocking=True), y.to(DEVICE,non_blocking=True)
            correct += (model(x).argmax(1)==y).sum().item()
            total   += len(y)
    return correct/total
print('Metrics ready.')


Metrics ready.


In [5]:
# ── Ablation A: CE-only 600 epochs ──────────────────────────────────
print('=== ABLATION A: CE-only 600 epochs ===')
print('Does NC1 reach <0.01 with cross-entropy alone?')
print()
torch.manual_seed(0)
model_ce = MLP(depth=5, width=512, act_cls=nn.ReLU).to(DEVICE)
try:
    model_ce = torch.compile(model_ce, mode='reduce-overhead')
except Exception:
    pass

opt = torch.optim.Adam(model_ce.parameters(), lr=1e-3, weight_decay=1e-4)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=600)
rows_ce = []; t0 = time.time(); terminal = False

for ep in range(1, 601):
    model_ce.train()
    for x, y in train_loader:
        x,y = x.to(DEVICE,non_blocking=True), y.to(DEVICE,non_blocking=True)
        opt.zero_grad(set_to_none=True)
        F.cross_entropy(model_ce(x), y).backward()
        opt.step()
    sch.step()
    if ep % 20 == 0 or ep == 600:
        tr = evaluate(model_ce, train_loader)
        te = evaluate(model_ce, test_loader)
        if tr >= 0.99 and not terminal:
            terminal = True
            print(f'  Terminal ep={ep}')
        nc = compute_nc(model_ce, train_loader) if terminal else \
             {'nc1':None,'nc2':None,'nc3':None,'feat_norm':None}
        rows_ce.append({'epoch':ep,'train':tr,'test':te,**nc})
        nc1s = f"{nc['nc1']:.5f}" if nc['nc1'] is not None else 'N/A'
        fns  = f"{nc['feat_norm']:.3f}" if nc['feat_norm'] else 'N/A'
        print(f'  ep={ep:>4} tr={tr:.4f} nc1={nc1s} fn={fns} '
              f't={(time.time()-t0)/60:.1f}m')

df_ce = pd.DataFrame(rows_ce)
df_ce.to_csv('/tmp/ce_only_ablation.csv', index=False)
nc_meas = df_ce.dropna(subset=['nc1'])
min_nc1 = nc_meas.nc1.min() if len(nc_meas) else float('inf')
final_fn = nc_meas.feat_norm.iloc[-1] if len(nc_meas) else None
print(f'\n=== RESULT ===')
print(f'Min NC1 in 600 CE epochs: {min_nc1:.5f}')
print(f'Final fn: {final_fn:.4f}' if final_fn else '')
if min_nc1 < 0.01:
    print('CE-only collapsed! Two-phase protocol is a speed-up, not a necessity.')
elif min_nc1 < 0.05:
    print('Partial collapse — CE alone cannot drive NC1 below 0.01.')
    print('Two-phase protocol justified.')
else:
    print('No collapse — two-phase protocol is necessary for NC1<0.01.')


=== ABLATION A: CE-only 600 epochs ===
Does NC1 reach <0.01 with cross-entropy alone?

  Terminal ep=20
  ep=  20 tr=0.9957 nc1=0.19635 fn=24.740 t=1.3m
  ep=  40 tr=0.9978 nc1=0.13266 fn=21.893 t=2.5m
  ep=  60 tr=0.9964 nc1=0.12456 fn=17.255 t=3.7m
  ep=  80 tr=0.9996 nc1=0.09145 fn=18.420 t=4.9m
  ep= 100 tr=0.9989 nc1=0.10106 fn=15.333 t=6.1m
  ep= 120 tr=0.9990 nc1=0.09820 fn=15.720 t=7.3m
  ep= 140 tr=0.9994 nc1=0.08224 fn=14.509 t=8.5m
  ep= 160 tr=1.0000 nc1=0.07609 fn=14.940 t=9.7m
  ep= 180 tr=0.9984 nc1=0.08916 fn=14.715 t=10.9m
  ep= 200 tr=0.9995 nc1=0.08702 fn=14.580 t=12.1m
  ep= 220 tr=0.9991 nc1=0.09650 fn=14.091 t=13.3m
  ep= 240 tr=0.9989 nc1=0.09405 fn=12.758 t=14.5m
  ep= 260 tr=0.9933 nc1=0.10374 fn=11.410 t=15.7m
  ep= 280 tr=0.9973 nc1=0.09295 fn=12.321 t=16.9m
  ep= 300 tr=1.0000 nc1=0.08660 fn=13.637 t=18.1m
  ep= 320 tr=1.0000 nc1=0.08450 fn=13.942 t=19.3m
  ep= 340 tr=1.0000 nc1=0.08240 fn=14.013 t=20.5m
  ep= 360 tr=1.0000 nc1=0.08261 fn=13.570 t=21.7m
  ep

In [6]:
# ── Ablation B: GELU seeds 3 and 4 ──────────────────────────────────
print('\n=== ABLATION B: GELU seeds 3 + 4 ===')
print('Existing: seeds 1,2 collapsed at T_NC=250, fn≈1.74, fn≈1.59')
print('Goal: find one more seed → remove dagger from Table 2')
print()

gelu_new = []
for seed in [3, 4]:
    print(f'--- GELU seed={seed} ---')
    torch.manual_seed(seed)
    model_g = MLP(depth=5, width=512, act_cls=nn.GELU).to(DEVICE)
    try:
        model_g = torch.compile(model_g, mode='reduce-overhead')
    except Exception:
        pass
    terminal = False; rows_g = []; t0 = time.time(); found = False
    for phase, loss_fn, n_ep in [(1,'ce',200),(2,'mse',500)]:
        opt_g = torch.optim.Adam(model_g.parameters(), lr=1e-3, weight_decay=1e-4)
        sch_g = torch.optim.lr_scheduler.CosineAnnealingLR(opt_g, T_max=n_ep)
        off = 200 if phase==2 else 0
        for ep_l in range(1, n_ep+1):
            ep = off + ep_l
            model_g.train()
            for x, y in train_loader:
                x,y = x.to(DEVICE,non_blocking=True), y.to(DEVICE,non_blocking=True)
                opt_g.zero_grad(set_to_none=True)
                logits = model_g(x)
                loss = (F.mse_loss(logits, F.one_hot(y,10).float())
                        if loss_fn=='mse' else F.cross_entropy(logits, y))
                loss.backward(); opt_g.step()
            sch_g.step()
            if ep_l % 10 == 0 or ep_l == n_ep:
                tr = evaluate(model_g, train_loader)
                if tr >= 0.99 and not terminal:
                    terminal = True
                nc = compute_nc(model_g, train_loader) if terminal else \
                     {'nc1':None,'nc2':None,'nc3':None,'feat_norm':None}
                rows_g.append({'epoch':ep,'train':tr,**nc})
                nc1s = f"{nc['nc1']:.5f}" if nc['nc1'] is not None else 'N/A'
                fns  = f"{nc['feat_norm']:.3f}" if nc['feat_norm'] else 'N/A'
                print(f'  ep={ep:>4} nc1={nc1s} fn={fns} t={(time.time()-t0)/60:.1f}m')
                if nc['nc1'] is not None and nc['nc1'] < 0.05 and nc['feat_norm'] < 5:
                    print(f'  *** COLLAPSED: T_NC={ep}  fn={nc["feat_norm"]:.4f}')
                    gelu_new.append({'seed':seed,'T_NC':ep,'fn':nc['feat_norm']})
                    pd.DataFrame(rows_g).to_csv(f'/tmp/gelu_s{seed}.csv', index=False)
                    found = True; break
                if nc['nc1'] is not None and nc['nc1'] > 5.0:
                    print(f'  NC1 diverging, skipping seed {seed}'); break
        if found: break

existing = [{'seed':1,'T_NC':250,'fn':1.897},{'seed':2,'T_NC':250,'fn':1.590}]
all_gelu = existing + gelu_new
df_gelu = pd.DataFrame(all_gelu)
df_gelu.to_csv('/tmp/gelu_summary.csv', index=False)
print('\n=== GELU FULL SUMMARY ===')
print(df_gelu.to_string())
if len(all_gelu) >= 3:
    fns = [r['fn'] for r in all_gelu]
    tncs = [r['T_NC'] for r in all_gelu]
    print(f'\n3 seeds: T_NC={sum(tncs)/len(tncs):.0f}  fn={sum(fns)/len(fns):.3f}')
    print('Remove dagger from Table 2. Change N from 2/3 to 3/3.')
else:
    print(f'\nOnly {len(all_gelu)} seeds. Keep dagger. Update N if needed.')



=== ABLATION B: GELU seeds 3 + 4 ===
Existing: seeds 1,2 collapsed at T_NC=250, fn≈1.74, fn≈1.59
Goal: find one more seed → remove dagger from Table 2

--- GELU seed=3 ---
  ep=  10 nc1=0.29715 fn=26.074 t=0.7m
  ep=  20 nc1=0.15555 fn=24.789 t=1.3m
  ep=  30 nc1=0.11451 fn=21.985 t=2.0m
  ep=  40 nc1=0.10094 fn=19.519 t=2.6m
  ep=  50 nc1=0.09023 fn=20.101 t=3.3m
  ep=  60 nc1=0.08742 fn=18.415 t=3.9m
  ep=  70 nc1=0.07367 fn=19.519 t=4.6m
  ep=  80 nc1=0.08170 fn=20.021 t=5.2m
  ep=  90 nc1=0.06880 fn=19.242 t=5.9m
  ep= 100 nc1=0.10632 fn=14.694 t=6.5m
  ep= 110 nc1=0.07785 fn=18.947 t=7.2m
  ep= 120 nc1=0.07767 fn=18.914 t=7.8m
  ep= 130 nc1=0.09516 fn=18.822 t=8.4m
  ep= 140 nc1=0.08658 fn=19.201 t=9.1m
  ep= 150 nc1=0.10341 fn=18.880 t=9.7m
  ep= 160 nc1=0.10965 fn=18.464 t=10.4m
  ep= 170 nc1=0.11500 fn=18.478 t=11.0m
  ep= 180 nc1=0.11907 fn=18.591 t=11.7m
  ep= 190 nc1=0.12102 fn=18.604 t=12.3m
  ep= 200 nc1=0.12147 fn=18.622 t=13.0m
  ep= 210 nc1=0.15657 fn=2.626 t=13.6m
  e

In [7]:
import os
for f in ['/tmp/ce_only_ablation.csv', '/tmp/gelu_summary.csv',
          '/tmp/gelu_s3.csv', '/tmp/gelu_s4.csv']:
    if os.path.exists(f): files.download(f)
print('Done.')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done.
